# Base Modeling

Este notebook parte del **dataset final** de Data Preparation (`dataset_final.parquet`, guardado en la última celda de `DataPreparation.ipynb`).

Aquí se hacen la normalización de los datos y la definición de los folds de validación con `TimeSeriesSplit`.

### 1. Conectar con Google Drive e importar librerías

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!rm -rf vms

In [3]:
import sys
!git clone https://github.com/kamimartinez/VMS-prediction.git vms
sys.path.append(os.path.abspath("vms/notebooks/modeling"))

Cloning into 'vms'...
remote: Enumerating objects: 334, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 334 (delta 53), reused 68 (delta 34), pack-reused 230 (from 1)
Receiving objects: 100% (334/334), 16.64 MiB | 22.60 MiB/s, done.
Resolving deltas: 100% (131/131), done.


In [4]:
!ls vms/notebooks

data  evaluation  modeling


### 2. Cargar el dataset final
Se lee desde `Datos/Procesado`. Si no existe, primero hay que ejecutar `DataPreparation.ipynb` completo.

In [5]:
RUTA_DATOS = "/content/drive/My Drive/Datos/Procesado"

def load_dataset(file="dataset_final_serie_tiempo.parquet"):
  dataset_final = pd.read_parquet(os.path.join(RUTA_DATOS, file))
  dataset_final["bloque_tiempo"] = pd.to_datetime(dataset_final["bloque_tiempo"])
  dataset_final = dataset_final.sort_values(["bloque_tiempo", "ms"]).reset_index(drop=True)

  print(f"dataset_final: {len(dataset_final):,} bloques x {dataset_final.shape[1]} columnas")
  print(list(dataset_final.columns))
  return dataset_final

### 3. Parámetros

In [6]:
TARGET = "pct_posible_cola"
N_FOLDS = 5
tscv = TimeSeriesSplit(n_splits=N_FOLDS)

# Normalización de datos

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
import pandas as pd

TARGET = "pct_posible_cola"

datasets = ["dataset_final_regresion.parquet", "dataset_final_serie_tiempo.parquet"]

df_final = load_dataset(datasets[1])
df_model = df_final.copy()

columnas_excluir = [TARGET, "ms", "bloque_tiempo"]

X = df_model.drop(columns=columnas_excluir)
y = df_model[TARGET]

categoricas = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
numericas = X.select_dtypes(include=["number"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categoricas),
    ],
    remainder="drop",
    verbose_feature_names_out=False,   # quita los prefijos num__ / cat__
)

# Que devuelva un DataFrame directamente (sklearn >= 1.2)
preprocessor.set_output(transform="pandas")

X_normalizado = preprocessor.fit_transform(X)

dataset_final: 179,850 bloques x 24 columnas
['ms', 'bloque_tiempo', 'total_ordenios', 'produccion_promedio', 'duracion_total_min', 'patadas_totales', 'desviaciones_leche_totales', 'ordenios_con_lavado', 'pct_primer_parto', 'pct_fresca', 'pct_alta', 'pct_baja', 'pct_extendida', 'hubo_ordenios', 'pct_ocupacion_robot', 'hubo_lavado', 'ordenios_otros_robots', 'hora', 'dia_semana', 'temporada', 'min_desde_alimento', 'vacas_activas_dia', 'sin_etapa', 'pct_posible_cola']


# Separación de train y tests

In [8]:
no_normalizadas = df_model[columnas_excluir]   # TARGET, ms, bloque_tiempo

# Reintegrar por índice
df_final = pd.concat([no_normalizadas, X_normalizado], axis=1)

def split_df(df_final, shuffle=False):
  df_train, df_temp = train_test_split(df_final, test_size=0.3, shuffle=shuffle)
  df_val, df_test = train_test_split(df_temp, test_size=0.5, shuffle=shuffle)
  return df_train, df_val, df_test

> **Nota:** en esta celda el `preprocessor` se ajusta con todo el dataset, lo que deja ver información del periodo de prueba. Para entrenar y evaluar modelos se debe ajustar (`fit`) solo con los bloques de entrenamiento de cada fold y aplicar `transform` a los de prueba.

# Validación

Se usa el mismo `TimeSeriesSplit` de Data Preparation (5 folds), ahora sobre `dataset_final`: cada fold entrena con el pasado y prueba con el periodo siguiente.

In [9]:
df_train, df_val, df_test = split_df(df_final, False)
df_train = df_train.sort_values(["bloque_tiempo", "ms"]).reset_index(drop=True)

folds = []
for i, (idx_ent, idx_prueba) in enumerate(tscv.split(df_train), start=1):
    t = df_train["bloque_tiempo"]
    folds.append({
        "fold": i,
        "entrenamiento": f"{t.iloc[idx_ent[0]].date()} a {t.iloc[idx_ent[-1]].date()}",
        "prueba": f"{t.iloc[idx_prueba[0]].date()} a {t.iloc[idx_prueba[-1]].date()}",
        "bloques_entrenamiento": len(idx_ent),
        "bloques_prueba": len(idx_prueba),
    })
display(pd.DataFrame(folds))


,fold,entrenamiento,prueba,bloques_entrenamiento,bloques_prueba
0,1,2022-08-01 a 2022-12-24,2022-12-24 a 2023-05-19,20985,20982
1,2,2022-08-01 a 2023-05-19,2023-05-19 a 2023-10-12,41967,20982
2,3,2022-08-01 a 2023-10-12,2023-10-12 a 2024-03-05,62949,20982
3,4,2022-08-01 a 2024-03-05,2024-03-05 a 2024-07-29,83931,20982
4,5,2022-08-01 a 2024-07-29,2024-07-29 a 2024-12-22,104913,20982


In [ ]:
from modeling.model_base import BaseModel
from modeling.lstm_model import LSTM

model: BaseModel = LSTM()
X = model.prepare(df_train)

ModuleNotFoundError: No module named 'models'